In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- acryo_loader shared ---
FIX_ACRYO_CORR_MAX = np.array([0.9, 0.8, 0.7])
FIX_ACRYO_LOCAL_SHIFTS = np.array([[0.123, 0.234, 0.345], [0.456, 0.567, 0.678], [0.789, 0.891, 0.912]])
FIX_ACRYO_ROTVEC = np.array([[0.111111, 0.222222, 0.333333], [0.444444, 0.555555, 0.666666], [0.777777, 0.888888, 0.999999]])
FIX_ACRYO_LABELS = ["A", "B", "C"]
FIX_ACRYO_BASE_FEATURES_PD = pd.DataFrame({"label": ["a", "b", "c"]})
FIX_ACRYO_BASE_FEATURES_PL = pl.from_pandas(FIX_ACRYO_BASE_FEATURES_PD)

def _set_loader_self_pd():
    global self
    self = SimpleNamespace(molecules=SimpleNamespace(pos=np.zeros((3, 3)), features=FIX_ACRYO_BASE_FEATURES_PD.copy()))

def _set_loader_self_pl():
    global self
    self = SimpleNamespace(molecules=SimpleNamespace(pos=np.zeros((3, 3)), features=FIX_ACRYO_BASE_FEATURES_PL.clone()))

def _make_rotator():
    return SimpleNamespace(as_rotvec=lambda: FIX_ACRYO_ROTVEC)

def _make_mole_pd():
    return SimpleNamespace(features=FIX_ACRYO_BASE_FEATURES_PD.copy(), pos=np.zeros((3, 3)))

def _make_mole_pl():
    return SimpleNamespace(features=FIX_ACRYO_BASE_FEATURES_PL.clone(), pos=np.zeros((3, 3)))

# --- acryo_loader_corr_max_with_columns_migration ---
FIX_ACRYO_LOADER_CORR_MAX_WITH_COLUMNS_MIGRATION_CORR_MAX = FIX_ACRYO_CORR_MAX
FIX_ACRYO_LOADER_CORR_MAX_WITH_COLUMNS_MIGRATION_GET_FEATURES = lambda *a, **k: None
FIX_ACRYO_LOADER_CORR_MAX_WITH_COLUMNS_MIGRATION_LABELS = FIX_ACRYO_LABELS
FIX_ACRYO_LOADER_CORR_MAX_WITH_COLUMNS_MIGRATION_LOCAL_SHIFTS = FIX_ACRYO_LOCAL_SHIFTS
FIX_ACRYO_LOADER_CORR_MAX_WITH_COLUMNS_MIGRATION_MOLE_ALIGNED = _make_mole_pd()
FIX_ACRYO_LOADER_CORR_MAX_WITH_COLUMNS_MIGRATION_ROTATOR = _make_rotator()

# --- acryo_loader_get_features_migration ---

# --- acryo_loader_scores_with_columns_migration ---
FIX_ACRYO_LOADER_SCORES_WITH_COLUMNS_MIGRATION_GET_FEATURES = lambda *a, **k: None
FIX_ACRYO_LOADER_SCORES_WITH_COLUMNS_MIGRATION_LOCAL_SHIFTS = FIX_ACRYO_LOCAL_SHIFTS
FIX_ACRYO_LOADER_SCORES_WITH_COLUMNS_MIGRATION_MOLE_ALIGNED = _make_mole_pd()
FIX_ACRYO_LOADER_SCORES_WITH_COLUMNS_MIGRATION_ROTATOR = _make_rotator()
FIX_ACRYO_LOADER_SCORES_WITH_COLUMNS_MIGRATION_SCORES = FIX_ACRYO_CORR_MAX
FIX_ACRYO_LOADER_SCORES_WITH_COLUMNS_MIGRATION_UPDATE_FEATURES = lambda features, values: features

# --- acryo_loader_update_features_migration ---

_set_loader_self_pd()
print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_acryo_loader_corr_max_with_columns_migration(corr_max, get_features, labels, local_shifts, mole_aligned, rotator):
    mole_aligned.features = pd.concat(
        [
            self.molecules.features,
            get_features(corr_max, local_shifts, rotator.as_rotvec()),
            pd.DataFrame({"labels": labels}),
        ],
        axis=1,
    )
    return None

def before_acryo_loader_get_features_migration():
    def get_features(corr_max, local_shifts, rotvec) -> pd.DataFrame:
        import pandas as pd

        features = {
            "score": corr_max,
            "shift-z": np.round(local_shifts[:, 0], 2),
            "shift-y": np.round(local_shifts[:, 1], 2),
            "shift-x": np.round(local_shifts[:, 2], 2),
            "rotvec-z": np.round(rotvec[:, 0], 5),
            "rotvec-y": np.round(rotvec[:, 1], 5),
            "rotvec-x": np.round(rotvec[:, 2], 5),
        }
        return pd.DataFrame(features)
    return get_features

def before_acryo_loader_scores_with_columns_migration(get_features, local_shifts, mole_aligned, rotator, scores, update_features):
    mole_aligned.features = update_features(
        self.molecules.features.copy(),
        get_features(scores, local_shifts, rotator.as_rotvec()),
    )
    return None

def before_acryo_loader_update_features_migration():
    def update_features(
        features: pd.DataFrame,
        values: dict | pd.DataFrame,
    ):
        for name, value in values.items():
            features[name] = value
        return features
    return update_features

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_acryo_loader_corr_max_with_columns_migration(corr_max, get_features, labels, local_shifts, mole_aligned, rotator):

    mole_aligned.features = pl.concat(
        [
            self.molecules.features,
            get_features(corr_max, local_shifts, rotator.as_rotvec()),
            pl.DataFrame({"labels": labels}),
        ],
        how="horizontal",
    )
    return None

def gen_acryo_loader_get_features_migration():
    import numpy as np


    def get_features(corr_max, local_shifts, rotvec) -> pl.DataFrame:
        features = {
            "score": corr_max,
            "shift-z": np.round(local_shifts[:, 0], 2),
            "shift-y": np.round(local_shifts[:, 1], 2),
            "shift-x": np.round(local_shifts[:, 2], 2),
            "rotvec-z": np.round(rotvec[:, 0], 5),
            "rotvec-y": np.round(rotvec[:, 1], 5),
            "rotvec-x": np.round(rotvec[:, 2], 5),
        }
        return pl.DataFrame(features)
    return get_features

def gen_acryo_loader_scores_with_columns_migration(get_features, local_shifts, mole_aligned, rotator, scores, update_features):
    mole_aligned.features = update_features(
        self.molecules.features.clone(),
        get_features(scores, local_shifts, rotator.as_rotvec()),
    )
    return None

def gen_acryo_loader_update_features_migration():


    def update_features(
        features: pl.DataFrame,
        values: dict | pl.DataFrame,
    ):
        for name, value in values.items():
            if isinstance(value, pl.Series):
                features = features.with_columns(value.alias(name))
            else:
                is_sequence = False
                if not isinstance(value, (str, bytes, dict)):
                    try:
                        len(value)
                        is_sequence = True
                    except TypeError:
                        is_sequence = False

                if is_sequence:
                    features = features.with_columns(pl.Series(name, value))
                else:
                    features = features.with_columns(pl.lit(value).alias(name))
        return features
    return update_features

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: acryo_loader_update_features_migration ===

try:
    _r = gen_acryo_loader_update_features_migration()(FIX_ACRYO_BASE_FEATURES_PL.clone(), {"score": [0.9, 0.8, 0.7]})
    print("✅ L1 smoke gen_acryo_loader_update_features_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_acryo_loader_update_features_migration: {type(_e).__name__}: {_e}")

try:
    _rb = before_acryo_loader_update_features_migration()(FIX_ACRYO_BASE_FEATURES_PD.copy(), {"score": [0.9, 0.8, 0.7]})
    print("✅ L1 smoke before_acryo_loader_update_features_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_acryo_loader_update_features_migration: {type(_e).__name__}: {_e}")

try:
    _rb = before_acryo_loader_update_features_migration()(FIX_ACRYO_BASE_FEATURES_PD.copy(), {"score": [0.9, 0.8, 0.7]})
    _rg = gen_acryo_loader_update_features_migration()(FIX_ACRYO_BASE_FEATURES_PL.clone(), {"score": [0.9, 0.8, 0.7]})
    compare(_rb, _rg, "acryo_loader_update_features_migration")
except Exception as _e:
    print(f"❌ L2 equivalence acryo_loader_update_features_migration: setup error — {type(_e).__name__}: {_e}")

# AUDIT-156: compare alternate-column update values.
try:
    _rb = before_acryo_loader_update_features_migration()(FIX_ACRYO_BASE_FEATURES_PD.copy(), {"quality": [1, 2, 3]})
    _rg = gen_acryo_loader_update_features_migration()(FIX_ACRYO_BASE_FEATURES_PL.clone(), {"quality": [1, 2, 3]})
    compare(_rb, _rg, "L3 edge acryo_loader_update_features_migration alternate column", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge acryo_loader_update_features_migration: {type(_e).__name__}: {_e}")
